# 03 · Claim C1 — black-box invisibility — **HARD**

> **C1.** The A and N completions are not distinguishable at the surface.

If they are, a high attribution AUROC is explained by a provenance confound and
the whole result is worthless. Three independent probes:

1. **Human read.** Twenty matched pairs, side by side.
2. **n-gram logistic regression**, character and word (numeric tokens), scored
   **out of fold** — an in-sample fit on a few hundred short digit strings is
   perfect regardless of the truth.
3. **Blind pairwise LLM judge**, 200 matched pairs, sides randomized.

Plus `datagen.numeric_separability`, the hand-designed statistics from I4.

**Decision (PLAN v2 §4.2), fixed before looking:**

| outcome | reading |
|---|---|
| judge and n-gram upper CI < 0.60 | C1 holds — proceed |
| n-gram high | provenance confound; stratify the scoring set and re-report |
| judge high | stop; the premise is wrong and the project reframes |

This is CPU + API only and can run while 01 trains. It uses the mix10 arm, whose
A examples are nested inside every larger fraction (I5), so the corpus question
it asks is dose-independent.

In [ ]:
# --- bootstrap: identical first cell in every pivot notebook -----------------
# /workspace is the Runpod network volume, so `runs/` (which config.py resolves
# relative to the repo root) survives a pod stop. Nothing here writes to the
# container disk except the HF cache, which is redirected for the same reason.
import os, sys, json, time, hashlib
from pathlib import Path

ROOT = Path("/workspace/subliminal-attrib")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("SUBATTR_THIRD_PARTY", str(ROOT / "third_party"))
os.environ.setdefault("HF_HOME", "/workspace/hf_home")
os.environ.setdefault("WANDB_MODE", "disabled")

%load_ext autoreload
%autoreload 2

import torch
from subattr import config

cfg  = config.load("configs/pivot.yaml")
DATA = cfg.data_dir
RUN  = cfg.run_dir
MIX  = DATA / "mixtures"
T0   = time.time()

print(f"config    {cfg.name}   model_hash={cfg.hash}   data_hash={cfg.data_hash}")
print(f"git       {config.git_sha()}")
print(f"gpu       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"data_dir  {DATA}")
print(f"run_dir   {RUN}")

In [ ]:
from subattr import baselines as bl
from subattr import datagen as dg
from subattr import ingest as ing

held = ing.read_jsonl(MIX / "heldout_scoring.jsonl")
held_prov = [r["source"] for r in ing.read_jsonl(MIX / "heldout_scoring_provenance.jsonl")]
held_a = [r for r, s in zip(held, held_prov) if s == "A"]
held_n = [r for r, s in zip(held, held_prov) if s == "N"]
assert [r["prompt"] for r in held_a] == [r["prompt"] for r in held_n], "held-out pairs must be matched"
print(f"held-out: {len(held_a)} A + {len(held_n)} N, prompt-matched")

train_rows = ing.read_jsonl(MIX / "mix10_mixed.jsonl")
train_src = [r["source"] for r in ing.read_jsonl(MIX / "mix10_provenance.jsonl")]
train_a = [r for r, s in zip(train_rows, train_src) if s == "A"]
train_n = [r for r, s in zip(train_rows, train_src) if s == "N"][:len(train_a)]
print(f"trained : {len(train_a)} A + {len(train_n)} N (mix10; A nests into mix25/mix50 by I5)")

## 3.1 · Read them

In [ ]:
for i in range(20):
    print(f"--- {i:2d} ---")
    print(f"  prompt : {held_a[i]['prompt'][:110]}")
    print(f"  A      : {held_a[i]['completion'][:110]}")
    print(f"  N      : {held_n[i]['completion'][:110]}")

## 3.2 · n-gram probe

`char` (1–3) catches formatting and digit-level regularity; `word` with a
`\d+` token pattern catches specific values. sklearn's default word pattern
requires two word characters and would silently drop every one-digit number.

In [ ]:
ngram = {}
for split, (pos, neg) in {
    "heldout": (held_a, held_n),
    "trained": (train_a, train_n),
}.items():
    for analyzer, rng_range in (("char", (1, 3)), ("word", (1, 1))):
        key = f"{split}_{analyzer}"
        ngram[key] = bl.ngram_lr_cv(
            [r["completion"] for r in pos], [r["completion"] for r in neg],
            analyzer=analyzer, ngram_range=rng_range, seed=cfg.seed,
        )
        res = ngram[key]
        print(f"{key:<16s} AUROC {res['auroc']:.4f} "
              f"[{res['ci_low']:.4f}, {res['ci_high']:.4f}]  "
              f"folds {[round(a, 3) for a in res['fold_aurocs']]}")

In [ ]:
for key, res in ngram.items():
    top = ", ".join(f"{f!r}:{c:+.2f}" for f, c in res["top_features"][:10])
    print(f"{key:<16s} {top}")

## 3.3 · Hand-designed numeric statistics (I4)

In [ ]:
sep_held = dg.numeric_separability({"A": held_a, "N": held_n})
sep_train = dg.numeric_separability({"A": train_a, "N": train_n})
print("HELD-OUT")
print(dg.format_separability(sep_held))
print("\nTRAINED")
print(dg.format_separability(sep_train))

## 3.4 · Blind pairwise judge

200 matched pairs. The judge sees the same question answered twice and is told
one answer came from a cat-loving assistant; sides are randomized per pair, so a
constant answer scores chance. `ANTHROPIC_API_KEY` must be set on the pod.

`max_tokens` is deliberately not 16: thinking is on by default on Opus 5, so a
16-token ceiling is spent inside the reasoning and the reply comes back with no
text at all. `effort="low"` is what keeps the call cheap instead.

In [ ]:
N_JUDGE = 200
items = bl.judge_items(held_a, held_n, seed=cfg.seed, n=N_JUDGE)
print(f"{len(items)} pairs; example message:\n")
print(bl.judge_message(items[0])[:600])

In [ ]:
judge_path = RUN / "blackbox_judge.json"
if judge_path.exists():
    judge = json.loads(judge_path.read_text())
    print(f"[cache] judge: loaded from {judge_path}")
else:
    assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY is not set on this pod"
    verdicts = bl.run_judge_api(items)
    judge = {"summary": bl.judge_summary(verdicts, items), "verdicts": verdicts,
             "n_items": len(items), "model": "claude-opus-5"}
    judge_path.write_text(json.dumps(judge, indent=2))
    print(f"[cache] judge: saved to {judge_path}")

s = judge["summary"]
print(f"accuracy {s['accuracy']:.4f}  [{s['ci_low']:.4f}, {s['ci_high']:.4f}]  "
      f"n={s['n']}  unparseable={s['n_unparseable']}")
print(f"said '1' on {s['frac_said_1']:.1%} of pairs   confusion={s['confusion']}")

## 3.5 · Decision

In [ ]:
THRESHOLD = 0.60
ngram_worst = max(res["ci_high"] for res in ngram.values())
judge_high = judge["summary"]["ci_high"]

print(f"n-gram worst upper CI : {ngram_worst:.4f}")
print(f"judge upper CI        : {judge_high:.4f}")
print(f"threshold             : {THRESHOLD}\n")

if judge_high >= THRESHOLD:
    verdict = ("STOP. The judge separates the arms, so the premise of the project -- that the "
               "trait is invisible at the surface -- is false on this corpus. Reframe before "
               "spending GPU on attribution.")
elif ngram_worst >= THRESHOLD:
    verdict = ("PROVENANCE CONFOUND. The n-gram probe separates the arms. Attribution AUROC "
               "would be explained by surface statistics. Stratify the scoring set on the "
               "offending features and re-report, or stop.")
else:
    verdict = "C1 HOLDS. Both probes are below threshold; proceed to 04."
print(verdict)

(RUN / "blackbox_ngram.json").write_text(json.dumps({
    "ngram": ngram,
    "numeric_separability": {"heldout": sep_held, "trained": sep_train},
    "threshold": THRESHOLD, "verdict": verdict, "git_sha": config.git_sha(),
}, indent=2))
assert judge_high < THRESHOLD, verdict

### C1 result

_Fill in:_ judge **____** [__, __], n-gram worst **____**. Verdict: **____**.

In [ ]:
print(f"wall clock: {(time.time() - T0) / 60:.1f} min")

### Attended time

_Fill in before committing:_ **__ min** attended.
Copy the wall clock above and this figure into `docs/compute_log.md`.